In [2]:
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [3]:
# Create a sample text file for Guide to Dia
text_content = """
Diabetes Risk and Screening Guidelines

1. Age Factor: Individuals aged 35 years and older are at a higher risk.
2. Body Mass Index: A BMI of 25 or higher indicates increased risk.
3. Blood Pressure: High blood pressure is a major risk factor.
4. Screening Recommendation: If you have two or more risk factors, you should take a Fasting Plasma Glucose test at a nearby clinic.
5. Medical Disclaimer: I am an AI, not a doctor. Please consult a clinic for a real diagnosis.
"""

# Save it to the raw folder
file_path = RAW_DIR / "diabetes_guidelines.txt"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(text_content)

print(f"Created file: {file_path}")

Created file: /content/week3_project/data/raw/diabetes_guidelines.txt


In [4]:
# Cell 3 - Verify uploaded inputs
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 1
diabetes_guidelines.txt bytes: 474
Upload verified. Select your relevant input(s) for Step A3.


In [5]:
# Cell 4 - Source Selection and Lineage Utilities
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected

    matches = sorted(
        p for p in RAW_DIR.rglob("**")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully. Source selection utilities are ready.")

Cell 4 completed successfully. Source selection utilities are ready.


In [6]:
# Cell 4A - Common Lineage Registry
from pathlib import Path
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")

    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")

    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")

    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print("Registered: ", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


In [7]:
# Cell 5A1 - Pathway A1 for Text
import re

# Select the exact source file
f = select_source(relative_path="diabetes_guidelines.txt")

# Extract text
text = f.read_text(encoding="utf-8")
page_count = None

if not text.strip():
    raise ValueError("No extractable text found.")

print("Extracted Content")
print("Source:", f.name)
print("Characters:", len(text))
print("Preview:")
print(text[:200])

# Clean text
clean = re.sub(r"[ \t]+", " ", text)
clean = re.sub(r"\n{3,}", "\n\n", clean).strip()

# Define chunking functions
def fixed_chunks(text, size=150):
    return [text[i:i+size] for i in range(0, len(text), size)]

def paragraph_chunks(text, size=150):
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", text):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

# Compare two preparation approaches
fixed = fixed_chunks(clean, size=150)
structured = paragraph_chunks(clean, size=150)

print("\nChunking Comparison")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))
print("\nFixed sample:")
print(repr(fixed[0][:150]))
print("\nParagraph-aware sample:")
print(repr(structured[0][:150]))
print("\nPathway A1 completed successfully.")

Extracted Content
Source: diabetes_guidelines.txt
Characters: 474
Preview:

Diabetes Risk and Screening Guidelines

1. Age Factor: Individuals aged 35 years and older are at a higher risk. 
2. Body Mass Index: A BMI of 25 or higher indicates increased risk.
3. Blood Pressure

Chunking Comparison
Fixed chunks: 4
Paragraph-aware chunks: 4

Fixed sample:
'Diabetes Risk and Screening Guidelines\n\n1. Age Factor: Individuals aged 35 years and older are at a higher risk. \n2. Body Mass Index: A BMI of 25 or h'

Paragraph-aware sample:
'1. Age Factor: Individuals aged 35 years and older are at a higher risk. \n2. Body Mass Index: A BMI of 25 or higher indicates increased risk.\n3. Blood'

Pathway A1 completed successfully.


In [8]:
# Cell 5A2 - Pathway A2
import json

# Choose the preparation approach
CHOSEN_METHOD = "paragraph-aware"

if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-150"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-150"
else:
    raise ValueError("Choose paragraph-aware or fixed")

# Create traceable records
records = []
for i, chunk in enumerate(chosen, 1):
    records.append({
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    })

# Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Exported {len(records)} records to {prepared_path}")

Exported 4 records to /content/week3_project/data/processed/prepared_text.jsonl


In [9]:
# B3 Basic output check
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: file exists and is non-empty")

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 1402
Validation: file exists and is non-empty


In [10]:
# Register Pathway A - PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered:  diabetes_guidelines.txt -> prepared_text.jsonl


In [11]:
# Cell 6A - Generate Common Lineage Manifest
assert lineage_entries, "No lineage entries registered. Register at least one selected pathway first."

lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1


In [12]:
# Cell 6B - Verify Common Lineage Manifest
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
    saved_records = saved_manifest.get("records", [])
    assert saved_records, "No lineage records found in saved manifest."
    assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match."

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))

COMMON LINEAGE VALIDATION: PASS
Verified records: 1


In [13]:
# Cell 7 - Common V1 Pipeline Utilities
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR/"data"/"raw"
PIPE_DIR = BASE_DIR/"data"/"pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR/relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)
    if kind == "images":
        inputs = sorted(p for p in source.iterdir() if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]

    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")

    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs]
    }

    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": "Not verified"
    }

    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready.")

Cell 7 ready.


In [14]:
# Cell 8A - Conditional Required for PDF/TXT
import re
import json

# Select the source and use the same settings as Part B
SOURCE_A = resolve_source("diabetes_guidelines.txt")
CONFIG_A = {
    "chunk_size": 150,
    "method": "paragraph-aware"
}

def inspect_documents(inputs, cfg):
    src = inputs[0]
    text = src.read_text(encoding="utf-8")
    if not text.strip():
        raise ValueError("No extractable text found.")
    return {"source": src.name, "characters": len(text), "pages": None}

def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    text = src.read_text(encoding="utf-8")
    text = re.sub(r"[\t]+", "", text).strip()
    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if method == "fixed":
        chunks = [text[i:i + size] for i in range(0, len(text), size)]
    elif method == "paragraph-aware":
        chunks = []
        current = ""
        for section in re.split(r"\n\s*\n", text):
            section = section.strip()
            if not section:
                continue
            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""
                chunks.extend(section[i:i + size] for i in range(0, len(section), size))
            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += ("\n\n" if current else "") + section
            else:
                if current:
                    chunks.append(current)
                current = section
        if current:
            chunks.append(current)

    if not chunks:
        raise ValueError("No text chunks produced")

    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {"id": i, "source": str(src), "text": chunk, "page": None}
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    mapping = [{"source": str(src), "output": str(target)}]
    return [target], mapping, method

# Execute the pipeline
run_A = register_and_run("documents", SOURCE_A, inspect_documents, prepare_documents, CONFIG_A)

SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json


In [15]:
# Cell 9 - Repeatability and Provenance Verification
assert PIPELINE_RUNS, "No pipeline runs registered."

print("-- Repeatability Verification --")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()

    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Source-to-output mapping changed for {kind}"

    print(f"PASS: {kind} - source fingerprints, prepared outputs and mappings match")

print("\nAll selected pipelines passed repeatability smoke checks.")

-- Repeatability Verification --
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents - source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [16]:
import shutil
from google.colab import files

shutil.make_archive('/content/week3_artifacts', 'zip', '/content/week3_project')
files.download('/content/week3_artifacts.zip')
print("ZIP file downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ZIP file downloaded.
